In [1]:
print("hello")

hello


In [2]:
import torch
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM(GB):",round(torch.cuda.get_device_properties(0).total_memory/1e9, 1))

CUDA: True
GPU: Tesla T4
VRAM(GB): 15.6


In [3]:
!pip install unsloth


  Using cached unsloth-2026.7.2-py3-none-any.whl.metadata (61 kB)
  Using cached unsloth_zoo-2026.7.2-py3-none-any.whl.metadata (33 kB)
  Using cached tyro-1.0.15-py3-none-any.whl.metadata (12 kB)
  Using cached xformers-0.0.35-py39-none-manylinux_2_28_x86_64.whl.metadata (1.2 kB)
  Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
  Using cached datasets-4.3.0-py3-none-any.whl.metadata (18 kB)
  Using cached hf_transfer-0.1.9-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.7 kB)
  Using cached transformers-5.5.0-py3-none-any.whl.metadata (32 kB)
  Using cached trl-0.24.0-py3-none-any.whl.metadata (11 kB)
  Using cached dill-0.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached torchao-0.17.0-cp310-abi3-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (20 kB)
  Using cached cut_cross_entropy-25.1.1-py3-none-any.whl.metadata (9.3 kB)
  Using cached msgspec-0.21.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manyl

In [4]:
from unsloth import FastLanguageModel
import torch
MAX_SEQ_LEN = 512
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length = MAX_SEQ_LEN,
    dtype = None,
    load_in_4bit = True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 32,
    lora_dropout = 0.05,
    bias = "none",
    target_modules = ["q_proj", "k_proj","v_proj","o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.7.2 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [11]:
from datasets import load_dataset
DATA_DIR = "/kaggle/input/datasets/kishanknaik/finetuning-banking-fy-25-26-dataset"
ds = load_dataset('json',data_files = {
    "train": f"{DATA_DIR}/train.jsonl",
    "val":f"{DATA_DIR}/val.jsonl",
})

set(["instruction","input","output","label"]).issubset(ds["train"].column_names), \
f"unexpected columns:{ds['train'].column_names}" 
print("labels seen :",set(ds["train"]["label"]))

EOS = tokenizer.eos_token
PROMPT = """Below is the instruction that describes a task, paired with an input, Write a response that appropriately completes the request
### Instruction:
{}
### Input:
{}
### Response:
{}
"""

def format_example(ex):
    return {"text":PROMPT.format(ex["instruction"],ex["input"],ex["output"])+EOS}
train_ds = ds["train"].map(format_example)
val_ds = ds["val"].map(format_example)
print(train_ds[0]["text"][:700])

labels seen : {'ADVICE_VIOLATION', 'NEEDS_DISCLAIMER', 'COMPLIANT', 'UNSUPPORTED'}


Map:   0%|          | 0/2174 [00:00<?, ? examples/s]

Map:   0%|          | 0/271 [00:00<?, ? examples/s]

Below is the instruction that describes a task, paired with an input, Write a response that appropriately completes the request
### Instruction:
Classify this financial claim for compliance.
### Input:
Muthoot Finance's gold loan portfolio increased to ₹67,200 crore in Q2 FY24, registering an 18% year-on-year growth as disclosed in their regulatory filings.
### Response:
COMPLIANT: The statement presents factual gold loan portfolio figures for Muthoot Finance and cites regulatory filings. Rewrite: Muthoot Finance's gold loan portfolio increased to ₹67,200 crore in Q2 FY24, registering an 18% year-on-year growth as disclosed in their regulatory filings.
<|im_end|>


In [38]:
from trl import SFTTrainer
from transformers import TrainingArguments
DRY_RUN = False

trainer =  SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = val_ds,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LEN,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps= 4,
        warmup_steps = 5,
        max_steps = 30 if DRY_RUN else -1,
        num_train_epochs = 1 if DRY_RUN else 3,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "/kaggle/working/outputs",
        save_steps = 200,
        save_strategy="no",
        eval_strategy="steps",
        eval_steps = 50,
        per_device_eval_batch_size = 4,
        average_tokens_across_devices= False,
        report_to = "none",
        
    ),
)

In [39]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "### Instruction:\n",
    response_part = "### Response:\n",
)

In [40]:
stats = trainer.train()
print(stats)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,174 | Num Epochs = 3 | Total steps = 204
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 4 x 1) = 32
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
50,0.732677,1.552304
100,0.554855,1.595907
150,0.392682,1.858253
200,0.386226,1.759234
204,0.386226,1.759172


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

TrainOutput(global_step=204, training_loss=0.5711263862310672, metrics={'train_runtime': 3486.1995, 'train_samples_per_second': 1.871, 'train_steps_per_second': 0.059, 'total_flos': 4.468958121040896e+16, 'train_loss': 0.5711263862310672, 'epoch': 3.0})


In [42]:
FastLanguageModel.for_inference(model)
prompt= PROMPT.format(
    "Classify this financial claim for compliance",
    "Reliance is guaranteed to cross 20 lakh crore market cap next year - buy now.",
    "",
)
inputs = tokenizer([prompt],return_tensors="pt").to("cuda")
out = model.generate(**inputs,max_new_tokens=96,use_cache=True)
print(tokenizer.decode(out[0][inputs.input_ids.shape[1]:],skip_special_tokens=True))

Both `max_new_tokens` (=96) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ADVICE_VIOLATION: The claim provides a direct purchase recommendation ('buy now') and guarantees a future market capitalization target. Rewrite: Reliance Industries' long-term growth potential is often evaluated by its current market capitalization relative to peers and industry trends.



In [47]:
from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
REPO = "kishanaik5/sentinel-compliance-qwen25-7b-lora"
model.save_pretrained("/kaggle/working/lora_adapter")
tokenizer.save_pretrained("/kaggle/working/lora_adapter")
model.push_to_hub(REPO,token=HF_TOKEN)
tokenizer.push_to_hub(REPO,token=HF_TOKEN)
print("pushed to https:huggingface.co/"+REPO)


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/lora_adapter/tokenizer_config.json.


README.md:   0%|          | 0.00/564 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/kishanaik5/sentinel-compliance-qwen25-7b-lora


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpb6_khema/tokenizer_config.json.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

pushed to https:huggingface.co/kishanaik5/sentinel-compliance-qwen25-7b-lora


In [46]:
import json, time
from sklearn.metrics import f1_score, classification_report
LABELS = ["COMPLIANT","NEEDS_DISCLAIMER","UNSUPPORTED","ADVICE_VIOLATION"]
def parse_label(text):
    t = text.strip().upper()
    for lab in LABELS:
        if t.startswith(lab): return lab
    for lab in lABELS:
        if lab in t : return lab
    return "UNKNOWN"
test = [json.loads(l) for l in open(f"{DATA_DIR}/test.jsonl")]
FastLanguageModel.for_inference(model)
preds,gold = [],[]
t0 = time.time()
for i,ex in enumerate(test):
    p = PROMPT.format(ex["instruction"],ex["input"],"")
    ids = tokenizer([p],return_tensors="pt").to("cuda")
    out = model.generate(**ids,max_new_tokens=16,use_cache=True)
    gen=tokenizer.decode(out[0][ids.input_ids.shape[1]:],skip_special_tokens=True)
    preds.append(parse_label(gen));gold.append(ex["label"])
    if (i+1)%50 == 0:
        print(f"{i+1}/{len(test)} ({time.time()-t0:0f}s")
macro_f1 = f1_score(gold,preds,labels=LABELS,average="macro")
print(f"\nMACRO-F1 = {macro_f1:3f} (Phase 4 target >=0.84)")
print(classification_report(gold,preds,labels=LABELS))

Both `max_new_tokens` (=16) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

50/271 (60.264924s


Both `max_new_tokens` (=16) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

100/271 (120.074045s


Both `max_new_tokens` (=16) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

...

200/271 (239.182659s


Both `max_new_tokens` (=16) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

250/271 (298.645569s


Both `max_new_tokens` (=16) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati


MACRO-F1 = 0.996323 (Phase 4 target >=0.84)
                  precision    recall  f1-score   support

       COMPLIANT       1.00      1.00      1.00        81
NEEDS_DISCLAIMER       0.99      1.00      0.99        68
     UNSUPPORTED       1.00      0.99      0.99        68
ADVICE_VIOLATION       1.00      1.00      1.00        54

        accuracy                           1.00       271
       macro avg       1.00      1.00      1.00       271
    weighted avg       1.00      1.00      1.00       271

